# Meshing in pyISSM — A Step-by-Step Guide

This notebook walks you through how to create a **mesh** for an ice sheet model using **pyISSM**, the Python interface for the Ice Sheet System Model (ISSM).

**No prior experience with meshing or modelling is assumed**. Each concept is explained in plain English before you run any code.

---

## What you'll do in this notebook

1. Understand what a mesh is and why it matters
2. Set up your working directories and import pyISSM
3. Create your first pyISSM model object
4. Build a simple structured mesh with `square_mesh`
5. Build a circular mesh with `round_mesh`
6. Build an unstructured mesh from a domain outline with `triangle`
7. Build a flexible adaptive mesh with `bamg` (uniform, non-uniform, and field-adapted)
8. Visualise and inspect your meshes throughout
9. Think about the philosophy behind creating meshes

---

> **How to use this notebook:**  
> - **Cells like this one** (formatted text) are explanations — read them before moving on.
> - **Code cells** (grey boxes with `[ ]:` on the left) are commands you run. Click on them and press **Shift + Enter**, or click the **▶ Run** button in the toolbar.
> - Work through the steps **in order** — later cells often depend on earlier ones!

> **Before starting:** Make sure you have completed the pyISSM installation notebook and that `import pyissm` runs without errors.

---

# PART 1 — What is a mesh and why do we need one?

---

## What is a mesh?

To simulate an ice sheet, ISSM needs to divide the ice sheet domain (the region of interest; a glacier or an entire ice sheet) into thousands of small **triangles**. This grid of triangles is called a **mesh**.

Think of it like pixelating a photograph: the more pixels (or in our case, triangles), the finer the detail you can capture. ISSM solves its ice physics equations at every **node** (corner point) of every triangle, so the mesh is the foundation of the entire model.

In finite element modelling (the basis underlying ISSM), each triangle is known as an **element** and the nodes of the triangles are referred to as **vertices**.

Here's why mesh design matters:

- **Too coarse** (big triangles) → fast to run, but misses important detail like grounding line dynamics
- **Too fine** (tiny triangles everywhere) → captures detail, but very slow and expensive to run
- **Smart mesh** → fine triangles where things change rapidly (e.g. near the grounding line), coarse triangles in slow-moving interior regions

pyISSM offers four main meshing functions, and we'll work through each of them:

| Function | Best for | Mesh type |
|---|---|---|
| `square_mesh` | Simple rectangular test domains | Structured (regular grid) |
| `round_mesh` | Circular test domains | Unstructured (irregular triangles) |
| `triangle` | Any domain shape defined by a boundary file | Unstructured, uniform |
| `bamg` | Real ice sheet domains needing variable resolution | Unstructured, anisotropic |

> **Structured vs unstructured?**  
> A structured mesh is like graph paper with triangles are arranged in a regular repeating pattern. An unstructured mesh places triangles more freely to better fit irregular shapes and to concentrate resolution where it's needed most. You may also hear unstructured meshes referred to as **anisotropic**, where the element size and shape varies.

---

# PART 2 — Setup

---

## Step 1: Import pyISSM and set up your directories

Before we do any meshing, we need to:
1. Import pyISSM and other useful Python packages
2. Define the paths to the folders we'll use for input files (assets) and saving our models (outputs)

**Update the `tutorial_dir` path below to point to your own project folder on Gadi.**

In [ ]:
import sys
import os
import pyissm
import numpy as np
import matplotlib.pyplot as plt

print("pyISSM imported successfully!")
print("pyISSM version:", pyissm.__version__)

In [ ]:
# ── Update this path to your own project directory on Gadi ──────────────────
tutorial_dir  = '/g/data/<project>/<userid>/pyISSM/tutorials'   # ← change <userid> to your userid
# ────────────────────────────────────────────────────────────────────────────

asset_dir     = tutorial_dir + '/assets'
execution_dir = tutorial_dir + '/models'

# Create the output directory if it doesn't already exist
if not os.path.isdir(execution_dir):
    os.mkdir(execution_dir)

print(f"tutorial_dir  : {tutorial_dir}")
print(f"asset_dir     : {asset_dir}")
print(f"execution_dir : {execution_dir}")

> **What are these directories for?**  
> - `asset_dir` is where your input files live — things like domain outline files (`.exp` files, explained later)
> - `execution_dir` is where you'll save model outputs
> - It's good practice to separate your inputs, outputs, and code into distinct folders

> **If you get a `FileNotFoundError`:** Check that `tutorial_dir` exists on Gadi and that you've replaced `<userid>` with your actual NCI username.

---

## Step 2: Create a new model object

Every ISSM simulation starts with a **model object** — an empty container that you fill up step by step: first a mesh, then physics parameters, then boundary conditions, and eventually a solution. In pyISSM, we create it with `pyissm.model.Model()` and store it in a variable called `md` (short for "model", by ISSM convention).

In [ ]:
# Create a new, empty model object
md = pyissm.model.Model()

print("Model object created!")

# Display the model — this shows all the fields available in md and their descriptions
md

> **What just happened?**  
> `md` is now a Python object with many sub-fields: `md.mesh`, `md.geometry`, `md.materials`, `md.initialization`, and so on. Right now most of them are empty — that's expected. Each step of building a model fills in more of these fields. The mesh section is what we'll be populating in this notebook.

> **One model object per mesher example:** Throughout this notebook we'll regularly create a fresh `md = pyissm.model.Model()` before each new meshing example, so we always start from a clean slate.

---

# PART 3 — `square_mesh`: A Simple Structured Mesh

---

## Step 3: Create a structured rectangular mesh

`square_mesh` is the simplest mesher in pyISSM. It generates a **structured, uniform mesh** over a rectangular domain, i.e. a regular grid of equally-sized triangles. You provide the size of the rectangle and how many nodes you want along each axis.

It's mainly used for idealised test problems (like ISMIP benchmark experiments) where a simple geometry is sufficient.

**Function signature:**
```python
pyissm.model.mesh.square_mesh(md, Lx, Ly, nx, ny)
```

| Argument | What it means |
|---|---|
| `md` | The model object |
| `Lx` | Width of the domain (metres) |
| `Ly` | Height of the domain (metres) |
| `nx` | Number of nodes along the x-axis |
| `ny` | Number of nodes along the y-axis |

In [ ]:
# Create a fresh model
md = pyissm.model.Model()

# square_mesh(model, x-length in m, y-length in m, nodes along x, nodes along y)
# Here: a 100,000m x 200,000m rectangle with 15 nodes along x and 25 along y
md = pyissm.model.mesh.square_mesh(md, Lx=100000, Ly=200000, nx=15, ny=25)

print("Square_mesh created!")

# Inspect what md.mesh now contains
md.mesh

In [ ]:
# Plot the mesh using pyISSM's built-in plotting function
fig, ax = pyissm.plot.plot_mesh2d(md,
                                  color='steelblue',
                                  linewidth=0.5,
                                  show_nodes=True,
                                  node_kwargs={'s': 15,
                                               'color': 'red',
                                               'alpha': 0.6})
ax.set_xlabel('X Coordinate (m)')
ax.set_ylabel('Y Coordinate (m)')
ax.set_title('square_mesh — 100km × 200km, 15×25 nodes')
plt.show()

print(f"Number of nodes    : {md.mesh.numberofvertices}")
print(f"Number of triangles: {md.mesh.numberofelements}")

Notice how the triangles are arranged in a very regular, repeating diagonal pattern. This is the "structured" nature of `square_mesh`. Every triangle is the same size.

---

## Step 4: Inspect the mesh object

After meshing, all the geometry information is stored inside `md.mesh`. Let's look at the most important fields:

In [ ]:
# md.mesh contains all the information about the mesh geometry.
# These are the fields you'll refer back to throughout all your modelling work.

print("=== Mesh summary ===")
print(f"Number of nodes (vertices)  : {md.mesh.numberofvertices}")
print(f"Number of triangles         : {md.mesh.numberofelements}")
print()
print(f"md.mesh.x — x-coordinate of every node")
print(f"  Shape: {md.mesh.x.shape}")
print(f"  First 5 values: {md.mesh.x[:5]}")
print()
print(f"md.mesh.y — y-coordinate of every node")
print(f"  Shape: {md.mesh.y.shape}")
print(f"  First 5 values: {md.mesh.y[:5]}")
print()
print(f"md.mesh.elements — connectivity table: each row lists 3 node indices for one triangle")
print(f"  Shape: {md.mesh.elements.shape}  (rows=triangles, columns=3 nodes per triangle)")
print(f"  First triangle: nodes {md.mesh.elements[0]}")

> **What are these fields?**  
> - `md.mesh.x` and `md.mesh.y` — the (x, y) coordinates of every node in the mesh, in metres
> - `md.mesh.elements` — the **connectivity matrix**: each row lists the 3 node indices that form one triangle. ISSM uses **1-based indexing** (node numbers start at 1, not 0).
> - `md.mesh.numberofvertices` — total number of nodes
> - `md.mesh.numberofelements` — total number of triangles

These fields are the same regardless of which mesher you use; they're the fundamental building blocks of every ISSM mesh.

---

### Try it yourself

Try increasing `nx` and `ny` to get a finer mesh, or decreasing them for a coarser one. What happens to the number of triangles?

In [ ]:
# Try changing nx and ny to explore how resolution affects the mesh
# For example, try nx=30, ny=50 for a finer mesh, or nx=5, ny=8 for very coarse

md_test = pyissm.model.Model()
md_test = pyissm.model.mesh.square_mesh(md_test, Lx=100000, Ly=200000,
                                         nx=30, ny=50)  # ← change these!

fig, ax = pyissm.plot.plot_mesh2d(md_test, color='steelblue', linewidth=0.5)
ax.set_title('square_mesh — try changing nx and ny')
ax.set_xlabel('X Coordinate (m)')
ax.set_ylabel('Y Coordinate (m)')
plt.show()

print(f"Nodes: {md_test.mesh.numberofvertices},  Triangles: {md_test.mesh.numberofelements}")
print()
print("   Notice: doubling nx and ny roughly quadruples the number of triangles!")
print("   More triangles = more accuracy, but also slower to run.")

---

# PART 4 — `round_mesh`: A Circular Mesh

---

## Step 5: Create a circular mesh

`round_mesh` generates a triangular mesh over a **circular domain**. It's used for idealised radially-symmetric experiments (like the EISMINT benchmark tests for ice sheet models). The triangles are placed irregularly to fill the circle. This is an **unstructured** mesh.

**Function signature:**
```python
pyissm.model.mesh.round_mesh(md, radius, resolution)
```

| Argument | What it means |
|---|---|
| `md` | The model object |
| `radius` | Radius of the circular domain (metres) |
| `resolution` | Target triangle edge length (metres) |

In [ ]:
# Create a fresh model
md = pyissm.model.Model()

# round_mesh(model, radius in m, resolution in m)
# Here: a circle of radius 800,000m (800km) with triangles ~50,000m (50km) across
md = pyissm.model.mesh.round_mesh(md, radius=800000, resolution=50000)

print("Round_mesh created!")
md.mesh

In [ ]:
# Plot the round mesh
fig, ax = pyissm.plot.plot_mesh2d(md,
                                  color='steelblue',
                                  linewidth=0.5,
                                  show_nodes=True,
                                  node_kwargs={'s': 15,
                                               'color': 'red',
                                               'alpha': 0.6})
ax.set_xlabel('X Coordinate (m)')
ax.set_ylabel('Y Coordinate (m)')
ax.set_title('round_mesh — radius 800km, resolution 50km')
plt.show()

print(f"Number of nodes    : {md.mesh.numberofvertices}")
print(f"Number of triangles: {md.mesh.numberofelements}")

The triangles are now irregular in shape — unlike `square_mesh`, there's no repeating pattern. Notice how the mesher fits triangles naturally into the circular boundary.

---

### Try it yourself

Try making the `resolution` smaller (finer mesh) or larger (coarser mesh). What's the minimum resolution you can use before it starts to take noticeably longer?

In [ ]:
# Try changing the resolution — smaller = finer mesh, larger = coarser mesh
md_test = pyissm.model.Model()
md_test = pyissm.model.mesh.round_mesh(md_test, radius=800000,
                                        resolution=100000)  # ← try 100000, 30000, 200000

fig, ax = pyissm.plot.plot_mesh2d(md_test, color='steelblue', linewidth=0.5)
ax.set_title('round_mesh — try changing resolution')
ax.set_xlabel('X Coordinate (m)')
ax.set_ylabel('Y Coordinate (m)')
plt.show()

print(f"Nodes: {md_test.mesh.numberofvertices},  Triangles: {md_test.mesh.numberofelements}")

---

# PART 5 — `triangle`: Meshing a Custom Domain Shape

---

## What is an ARGUS `.exp` file?

For `triangle` and `bamg`, you need to define the **shape of your domain**, characterised by the boundary (outline) of the region you want to mesh. pyISSM uses a file format called **ARGUS** for this, with the `.exp` file extension.

An ARGUS file is a plain text file listing the (x, y) coordinates of the domain boundary, tracing it like drawing around the outline with a pencil. The outline must be a **closed loop**, with the last coordinate matching the first.

Here's what one looks like for a square:

```
## Name:DomainOutline
## Icon:0
# Points Count  Value
5 1.000000
# X pos Y pos
0 0
500000 0
500000 500000
0 500000
0 0
```

For real ice sheet applications, the domain outline comes from satellite-derived ice sheet boundary datasets, and you'd generate the `.exp` file from that data. For this tutorial, we'll use the pre-made domain file from the tutorial assets.

> The tutorial assets folder (`asset_dir + '/Exp/'`) contains several example `.exp` files for you to use.

---

## Step 6: Mesh a domain with `triangle`

`triangle` is a fast, well-tested meshing algorithm. It takes your domain outline and fills it with high-quality, approximately uniformly-sized triangles.

**Function signature:**
```python
pyissm.model.mesh.triangle(md, domain_name, resolution)
```

| Argument | What it means |
|---|---|
| `md` | The model object |
| `domain_name` | Full path to the `.exp` domain outline file |
| `resolution` | Target triangle edge length (metres) |

In [ ]:
# Create a fresh model
md = pyissm.model.Model()

# triangle(model, path to domain outline file, resolution in metres)
md = pyissm.model.mesh.triangle(md,
                                domain_name=asset_dir + '/Exp/SquareIceShelf_DomainOutline.exp',
                                resolution=50000)

print("Triangle mesh created!")
md.mesh

In [ ]:
# Plot the triangle mesh
fig, ax = pyissm.plot.plot_mesh2d(md,
                                  color='steelblue',
                                  linewidth=0.5,
                                  show_nodes=True,
                                  node_kwargs={'s': 20,
                                               'color': 'red',
                                               'alpha': 0.5})
ax.set_xlabel('X Coordinate (m)')
ax.set_ylabel('Y Coordinate (m)')
ax.set_title('triangle mesh — Square Ice Shelf, resolution 50km')
plt.show()

print(f"Number of nodes    : {md.mesh.numberofvertices}")
print(f"Number of triangles: {md.mesh.numberofelements}")

The triangles are irregular but roughly similar in size across the domain. This is a **uniform unstructured mesh**. It's more flexible than `square_mesh` because it can fit any domain shape.

---

### Try it yourself by changing the resolution

In [ ]:
# Try a finer resolution — more triangles, slower to run but more accurate
md_test = pyissm.model.Model()
md_test = pyissm.model.mesh.triangle(md_test,
                                     domain_name=asset_dir + '/Exp/SquareIceShelf_DomainOutline.exp',
                                     resolution=20000)  # ← try 100000, 20000, 10000

fig, ax = pyissm.plot.plot_mesh2d(md_test, color='steelblue', linewidth=0.5)
ax.set_title('triangle mesh — try changing resolution')
ax.set_xlabel('X Coordinate (m)')
ax.set_ylabel('Y Coordinate (m)')
plt.show()

print(f"Nodes: {md_test.mesh.numberofvertices},  Triangles: {md_test.mesh.numberofelements}")

---

# PART 6 — `bamg`: The Powerful Adaptive Mesher

---

## What is BAMG and why is it special?

BAMG stands for **Bidimensional Anisotropic Mesh Generator**. It's the most powerful mesher in ISSM and the one you'll use most for real ice sheet modelling. Its key strength is that it can create **non-uniform meshes** — finer resolution in some areas, coarser in others — and it can automatically adapt to a data field (like ice velocity or thickness) to put fine resolution exactly where it's needed.

> Again, **anisotropic** means the triangles don't have to be equal in all directions. They can be stretched and elongated to follow the shape of a feature (e.g. the grounding line). This captures complex features with fewer triangles than a purely isotropic mesh would need.

**Function signature:**
```python
pyissm.model.mesh.bamg(md, **kwargs)
```

`bamg` takes keyword arguments. The most important ones are:

| Keyword | What it does |
|---|---|
| `domain` | Path to the domain outline `.exp` file (used when starting from scratch) |
| `hmax` | Maximum triangle edge length (metres) — sets the resolution ceiling |
| `hmin` | Minimum triangle edge length (metres) — prevents infinitely small triangles |
| `hVertices` | Array of per-node target resolutions for regional refinement |
| `field` | Data field to adapt the mesh to (e.g. velocity) |
| `err` | Interpolation error tolerance — smaller = finer mesh along sharp gradients |
| `gradation` | Controls how fast mesh size changes between neighbouring triangles |
| `anisomax` | Maximum allowed triangle elongation (1 = equilateral, higher = more elongated) |

---

## Step 7: A uniform BAMG mesh

First, let's use `bamg` to create a simple uniform mesh — similar to `triangle`, but using BAMG's algorithm.

In [ ]:
# Create a fresh model
md = pyissm.model.Model()

# bamg with a uniform resolution:
# 'domain' tells bamg where to find the domain outline
# 'hmax' sets the maximum edge length — this controls overall resolution
md = pyissm.model.mesh.bamg(md,
                             domain=asset_dir + '/Exp/SquareIceShelf_DomainOutline.exp',
                             hmax=50000)

print("BAMG uniform mesh created!")
md.mesh

In [ ]:
# Plot the BAMG uniform mesh
fig, ax = pyissm.plot.plot_mesh2d(md,
                                  color='steelblue',
                                  linewidth=0.5,
                                  show_nodes=True,
                                  node_kwargs={'s': 15,
                                               'color': 'red',
                                               'alpha': 0.5})
ax.set_xlabel('X Coordinate (m)')
ax.set_ylabel('Y Coordinate (m)')
ax.set_title('BAMG mesh — uniform, hmax=50km')
plt.show()

print(f"Number of nodes    : {md.mesh.numberofvertices}")
print(f"Number of triangles: {md.mesh.numberofelements}")

---

## Step 8: A non-uniform BAMG mesh, refined near specific vertices

In real ice sheet models you often want fine resolution near a specific area, for example near the grounding line, while keeping the rest of the mesh coarse to save computation time.

You can do this by specifying a resolution for each **vertex of the domain outline** using `hVertices`. Use `np.nan` for vertices where you want to keep the default (`hmax`) resolution.

The `SquareIceShelf_DomainOutline.exp` file has 4 corner vertices. Let's make one corner highly refined:

In [ ]:
# Create a fresh model
md = pyissm.model.Model()

# hVertices sets a target resolution at each vertex of the domain outline
# Our square outline has 4 corners. We want 3 of them coarse and 1 very fine.
# np.nan means "use the default hmax resolution for this vertex"
hvertices = np.array([np.nan,    # corner 1 — default (coarse)
                      np.nan,    # corner 2 — default (coarse)
                      5000,      # corner 3 — refined to 5km  ← highly refined!
                      np.nan])   # corner 4 — default (coarse)

md = pyissm.model.mesh.bamg(md,
                             domain=asset_dir + '/Exp/SquareIceShelf_DomainOutline.exp',
                             hmax=50000,
                             hVertices=hvertices)

print("BAMG non-uniform mesh created!")
print(f"Number of nodes    : {md.mesh.numberofvertices}")
print(f"Number of triangles: {md.mesh.numberofelements}")

In [ ]:
# Plot the non-uniform BAMG mesh
fig, ax = pyissm.plot.plot_mesh2d(md,
                                  color='steelblue',
                                  linewidth=0.5)
ax.set_xlabel('X Coordinate (m)')
ax.set_ylabel('Y Coordinate (m)')
ax.set_title('BAMG mesh — non-uniform, one refined corner')
plt.show()

You should see very small, dense triangles near the refined corner, gradually becoming larger as you move away from it. This is exactly what you'd want near a glacier terminus or grounding line in a real model.

> **In a real ice sheet model**, you'd refine near the grounding line (where ice transitions from grounded to floating) because that's where the most important and rapidly-changing dynamics occur.

---

## Step 9: Adapting the mesh to a data field

The most powerful feature of `bamg` is **mesh adaptation** — automatically refining the mesh where a data field (like ice velocity or thickness) changes most rapidly, and keeping it coarse where the field is smooth. This minimises error without making the whole mesh unnecessarily fine.

The typical workflow is:
1. Start with a coarse uniform mesh
2. Interpolate a data field onto the mesh nodes
3. Call `bamg` again with `field=` and `err=` — it analyses the field and refines where needed

> **Important:** When calling `bamg` for adaptation (step 3), you do **not** pass `domain=` again. BAMG adapts the mesh that already exists inside `md`.

For this example, we'll use observed velocity data from the tutorial assets:

---

# PART 7 — A Real-World Mesh: Adapting to Antarctic Data

---

## Step 9: Building a real-world mesh — the Wilkes Subglacial Basin

Now we'll work through a realistic meshing workflow for the Wilkes Subglacial Basin in East Antarctica. This is exactly the kind of thing you'd do at the start of a real ice sheet modelling project.

The strategy is:
1. Create an initial coarse mesh from the Wilkes domain outline using `triangle`
2. Load real Antarctic data from the **ACCESS Cryosphere Data Pool** — ice velocity (Mouginot 2017) and the BedMachine ice mask
3. Run a **refinement** — interpolating data onto the current mesh and using `bamg` to refine based on velocity
4. Add geographic coordinate information and save

---

## Step 10: Set up imports and the data catalog

We use the **ACCESS Cryosphere Data Pool** (`ccdtools`) to load both datasets. This tool knows exactly where the data lives on Gadi and loads it as an `xarray` Dataset, so you don't need to find or manage file paths yourself.

> **What is `ccdtools`?** It's the Python package for the ACCESS Cryosphere Community Datapool — a curated collection of Antarctic and Arctic datasets hosted on Gadi. You can browse available datasets with `catalog.search()` or `catalog` to see the full list.

In [ ]:
import ccdtools as dp

# Initialise the ACCESS Cryosphere Data Pool catalog
# This gives us access to all curated Antarctic datasets on Gadi
catalog = dp.catalog.DataCatalog()

# You can explore what's available by running: catalog
# Or search by keyword:                        catalog.search('velocity')
catalog

---

## Step 11: Generate the initial mesh with `triangle`

We start with a coarse `triangle` mesh at 5 km resolution over the Wilkes domain. This gives BAMG enough nodes to work with in the first refinement pass.

> **Why `triangle` first, not `bamg`?** `triangle` is faster for generating the initial uniform mesh. We then hand it to `bamg` for the data-adaptive refinement passes.

**Update the `domain_name` path below to point to the Wilkes domain outline file in your project directory.**

In [ ]:
# ── Update this path to your Wilkes domain outline file ─────────────────────
wilkes_exp = '/g/data/<project>/<userid>/wilkes/newWilkesDomain.exp'  # ← change <userid>
# ────────────────────────────────────────────────────────────────────────────

print('-- Generating first mesh')
md = pyissm.model.Model()

# triangle(model, domain outline file, resolution in metres)
# 5e3 = 5,000 m = 5 km — a reasonable starting resolution
md = pyissm.model.mesh.triangle(md,
                                domain_name=wilkes_exp,
                                resolution=5e3)

# Quick look at the initial mesh
fig, ax = pyissm.plot.plot_mesh2d(md, color='steelblue', linewidth=0.3)
ax.set_xlabel('X Coordinate (m)')
ax.set_ylabel('Y Coordinate (m)')
ax.set_title('Initial triangle mesh — Wilkes Subglacial Basin domain, 5 km')
plt.show()

print(f'Initial mesh: {md.mesh.numberofvertices} nodes, {md.mesh.numberofelements} triangles')

---

## Step 12: Load Antarctic velocity and BedMachine data from the data pool

We load two datasets from the ACCESS Cryosphere Data Pool:

- **BedMachine Antarctica v3** — provides an ice/ocean **mask** telling us what type of surface each point is:

| Mask value | Meaning |
|---|---|
| 0 | Open ocean |
| 1 | Ice-free land |
| 2 | Grounded ice |
| 3 | Floating ice (ice shelf) |

- **Mouginot et al. 2017 Antarctic velocity** — provides vx and vy velocity components across Antarctica

Both are loaded as lazy `xarray` Datasets and then interpolated onto the mesh nodes using `pyissm.data.interp.xr_to_mesh()`, which handles the coordinate transformation automatically.

> **Loading these datasets takes a moment** — they are large files (~GBs). The data pool loads them lazily using `dask`, so the actual data only arrives when `xr_to_mesh` requests it for your specific mesh coordinates.

In [ ]:
# Load BedMachine Antarctica v3 from the data pool
# This gives us the ice mask: 0=ocean, 1=ice-free land, 2=grounded ice, 3=floating ice
bedmachine_data = catalog.load_dataset('measures_bedmachine_antarctica', version='v3')

# Interpolate the mask onto our mesh node coordinates
# xr_to_mesh handles the interpolation from the regular grid onto our unstructured mesh
mask = pyissm.data.interp.xr_to_mesh(bedmachine_data, 'mask', md.mesh.x, md.mesh.y)

# Load Mouginot 2017 InSAR-based Antarctic velocity
velocity_data = catalog.load_dataset(
    'measures_insar_based_antarctica_ice_velocity_map', version='v2')

# Interpolate vx and vy onto mesh nodes, then compute ice speed
vx  = pyissm.data.interp.xr_to_mesh(velocity_data, 'VX', md.mesh.x, md.mesh.y)
vy  = pyissm.data.interp.xr_to_mesh(velocity_data, 'VY', md.mesh.x, md.mesh.y)
vel = np.sqrt(vx**2 + vy**2)  # speed in m/yr

print(f'mask range : {int(mask.min())} – {int(mask.max())}')
print(f'vel range  : {np.nanmin(vel):.1f} – {np.nanmax(vel):.1f} m/yr')

---

## Step 13: Run the bamg refinement

Now we run the refinement. 

1. Zeros out velocity where data is missing or where we're outside the ice sheet (`mask < 2`)
2. Builds **per-node resolution arrays** (`hmaxVertices`, `hminVertices`) to enforce tighter rules in specific regions:
   - **Grounded ice moving faster than 50 m/yr** → cap element size at 2 km
   - **Floating ice / ice shelves** → minimum element size of 2 km (fine resolution everywhere on the shelf, important for ice-ocean interactions)
3. Calls `bamg` to adapt the mesh using velocity as the metric field
4. Resets BAMG's internal state with `md.private.bamg = {}`
5. Re-interpolates velocity and mask onto the **new** mesh node positions (the nodes moved, so the old values are stale)

**Key `bamg` arguments used here:**

| Argument | Value | What it does |
|---|---|---|
| `field` | `vel` | The data field to adapt the mesh to |
| `err` | 1 | Interpolation error tolerance — tight, giving a well-refined mesh |
| `hmin` | 50 m | No triangle can be smaller than 50 m |
| `hmax` | 50,000 m | No triangle can be larger than 50 km |
| `hmaxVertices` | array | Per-node maximum size (fine on fast-flowing grounded ice) |
| `hminVertices` | array | Per-node minimum size (fine on ice shelves) |
| `maxnbv` | 2,000,000 | Hard cap on total nodes |
| `gradation` | 1.2 | Adjacent triangles can differ by at most a factor of 1.2 — smooth transitions |

> **Each pass takes a few minutes** on Gadi depending on the domain size and the resulting mesh complexity.

In [ ]:
# Refinement to improve the mesh

# Zero velocity where data is missing or outside the ice sheet (ocean / ice-free land)
# np.isnan catches nodes with no velocity coverage
# mask < 2 catches ocean (0) and ice-free land (1)
vel[np.isnan(vel) | (mask < 2)] = 0

# Build per-node resolution arrays
# NaN means "no special constraint — use the global hmin/hmax"
hmaxv = np.full(md.mesh.numberofvertices, np.nan)
hminv = np.full(md.mesh.numberofvertices, np.nan)

# Fast-flowing grounded ice (>50 m/yr, mask==2): cap element size at 2000 m
# These are the dynamically active regions where fine resolution matters most
pos_fast = np.where((vel > 50) & (mask == 2))[0]
hmaxv[pos_fast] = 2000

# Floating ice / ice shelves (mask==3): enforce minimum element size of 2000 m
# Ice shelves are where ocean-ice interaction happens
pos_float = np.where(mask == 3)[0]
hminv[pos_float] = 2000

print('   -- Remeshing with bamg')
md = pyissm.model.mesh.bamg(md,
                             field=vel,
                             err=1,
                             hmin=50,
                             hmax=50e3,
                             hmaxVertices=hmaxv,
                             hminVertices=hminv,
                             maxnbv=2e6,
                             gradation=1.2)

print(f'   Nodes: {md.mesh.numberofvertices} nodes, '
      f'{md.mesh.numberofelements} triangles')

print('\nRefinement complete!')

In [ ]:
# Visualise the final adapted mesh
fig, ax = pyissm.plot.plot_mesh2d(md, color='steelblue', linewidth=0.3)
ax.set_xlabel('X Coordinate (m)')
ax.set_ylabel('Y Coordinate (m)')
ax.set_title('Final Wilkes mesh, velocity-adapted, 2 refinement passes')
plt.show()

---

## Step 14: Add geographic coordinates

The mesh coordinates are in **polar stereographic metres** (EPSG:3031 — the standard projection for Antarctic work). We also store the **latitude and longitude** of each node and record the EPSG code.

`pyissm.tools.general.xy_to_ll()` converts from polar stereographic (x, y) to (latitude, longitude). The `-1` argument specifies the southern hemisphere.

In [ ]:
# Convert mesh node coordinates from polar stereographic (m) to lat/lon
# -1 specifies southern hemisphere
[md.mesh.lat, md.mesh.long] = pyissm.tools.general.xy_to_ll(md.mesh.x, md.mesh.y, -1)
md.mesh.epsg = 3031  # WGS 84 / Antarctic Polar Stereographic

print(f'Latitude range  : {md.mesh.lat.min():.2f}° → {md.mesh.lat.max():.2f}°')
print(f'Longitude range : {md.mesh.long.min():.2f}° → {md.mesh.long.max():.2f}°')
print(f'EPSG            : {md.mesh.epsg}')

In [ ]:
# Inspect the full mesh object
md.mesh

---

## Step 15: Save the mesh

`pyissm.model.io.save_model()` saves the model to a `.nc` (NetCDF) file. You can reload it later with `pyissm.model.io.load_model()`.

**Update the save path below to your own project directory.**

In [ ]:
# ── Update this path ─────────────────────────────────────────────────────────
save_path = '/g/data/<project>/<userid>/wilkes/wilkes_mesh.nc'  # ← change <userid>
# ─────────────────────────────────────────────────────────────────────────────

pyissm.model.io.save_model(md, save_path)
print(f'Model saved to: {save_path}')
print(f'To reload: md = pyissm.model.io.load_model("{save_path}")')

---

# You're done!

You've now worked through all four of pyISSM's main meshers and a complete real-world meshing workflow for Antarctic ice sheet modelling.

---

## Quick reference for meshing in pyISSM

| Function | Use case | Key arguments |
|---|---|---|
| `pyissm.model.mesh.square_mesh(md, Lx, Ly, nx, ny)` | Rectangular test domains | Domain size (m), nodes per axis |
| `pyissm.model.mesh.round_mesh(md, radius, resolution)` | Circular test domains | Radius (m), target element size (m) |
| `pyissm.model.mesh.triangle(md, domain_name, resolution)` | Any domain, uniform mesh | Path to `.exp` file, element size (m) |
| `pyissm.model.mesh.bamg(md, domain=..., hmax=...)` | Any domain, uniform BAMG mesh | Path to `.exp` file, max element size |
| `pyissm.model.mesh.bamg(md, domain=..., hmax=..., hVertices=...)` | Non-uniform, vertex-controlled | Per-vertex resolution array (use `np.nan` for default) |
| `pyissm.model.mesh.bamg(md, field=..., err=..., hmin=..., hmax=..., hmaxVertices=..., hminVertices=..., gradation=...)` | Adapt to a data field | Field array, error tolerance, per-node min/max size arrays |

**Essential `bamg` keywords:**
- `domain` — path to `.exp` outline file (**first call only**; omit when adapting an existing mesh)
- `hmax` / `hmin` — global max / min edge length in metres
- `hmaxVertices` / `hminVertices` — per-node max / min arrays (`np.nan` = no constraint)
- `maxnbv` — hard cap on total node count
- `field` — data field to adapt to (one value per node)
- `err` — interpolation error tolerance (smaller = finer along gradients)
- `gradation` — max size ratio between adjacent triangles

**Data pool pattern:**
```python
import ccdtools as dp
catalog = dp.catalog.DataCatalog()
ds = catalog.load_dataset('dataset_name', version='vX')
field_on_mesh = pyissm.data.interp.xr_to_mesh(ds, 'variable', md.mesh.x, md.mesh.y)
```

---

## Troubleshooting

| Problem | What to check |
|---|---|
| `IOError` on `.exp` file | Check your domain file path |
| `catalog.load_dataset` fails | Confirm the dataset name with `catalog.search()` |
| `xr_to_mesh` returns all NaN | Check that mesh coordinates are in EPSG:3031 (metres), not lat/lon |
| `bamg` second pass behaves unexpectedly | Always reset `md.private.bamg = {}` and re-interpolate data after each pass |
| Very slow meshing | Your `err` or `hmin` may be too tight — relax them and try again |

---

## Useful links

- **ISSM Mesh Tutorial**: https://issmteam.github.io/ISSM-Documentation/using-issm/tutorials/mesh.html
- **pyISSM documentation**: https://pyissm.readthedocs.io/latest/
- **ACCESS Cryosphere Data Pool**: https://access-nri.github.io/ccdtools/